# Step 0 · [2] Expansion: codes and words we are missing

The current rules were written by hand. Here the data suggests what to add, starting from the families that are **in the sector for sure (rule A)**:

- **A · Codes (lift).** CPC / IPC symbols that are frequent among the sector families but rare among all patents, and that our config does not cover yet.
- **B · Words (keyness).** 1–3-word phrases that appear in many more sector abstracts than in a random sample of patent abstracts.
- **C · Audio & hearing words.** Phrases typical of the audio / hearing families compared with the rest of the sector, for the Amplifon segment.

**Lift** of a code, and **keyness** of a phrase, use the same idea:

$$\text{lift} = \frac{\text{share of sector families with it}}{\text{share of all (or background) families with it}}$$

A lift of 100 means the code or phrase is 100 times more common in the sector than in patents in general. We use document shares (does a family have it or not) rather than TF-IDF weights, because they read directly as "x% of sector families".

The output is **a list of suggestions, not a decision**: read each candidate (codes in the CPC scheme, phrases in a few abstracts) and add the good ones to `config/sector.yaml`. Then re-run `02_candidate_pool` and `03_texts`.

**Before running:** `03_texts` with the same `ENV`.

In [ ]:
%pip install -q pyyaml scikit-learn

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sqlalchemy import distinct, func

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN, TLS202_APPLN_TITLE, TLS203_APPLN_ABSTR, TLS209_APPLN_IPC, TLS224_APPLN_CPC,
)

from src.sector import configured_codes, load_config, scope_filter, sector_family_ids

ENV = "TEST"                                   # same environment as 03_texts
OUTPUT_DIR = Path("outputs/step0")
pd.set_option("display.max_colwidth", 120)

# Thresholds: smaller on TEST, which is a tiny sample of PATSTAT
MIN_CODE_FAMILIES = 30 if ENV == "PROD" else 1     # a suggested code must be on at least this many sector families
MIN_PHRASE_DF = 20 if ENV == "PROD" else 1         # a phrase must appear in at least this many documents
BACKGROUND_MOD = 1000 if ENV == "PROD" else 1      # background sample: applications with appln_id % MOD == 0
BACKGROUND_N = 30000                               # at most this many background abstracts
TOP = 40                                           # rows shown in each table

cfg = load_config()
sel = cfg["selection"]
patstat = PatstatClient(env=ENV)
db = patstat.orm()


def read_table(stem):
    """Read outputs/step0/<stem>_<ENV> as parquet, or as CSV if parquet was not available."""
    path = OUTPUT_DIR / f"{stem}_{ENV}.parquet"
    return pd.read_parquet(path) if path.exists() else pd.read_csv(path.with_suffix(".csv"))

## A · Codes: lift

For every CPC (and IPC) symbol carried by the sector families, the number of sector families with it and the number of **all** families (2000–2025, patents) with it. Both counts run in the database.

In [ ]:
sector_q = sector_family_ids(db, cfg)
n_sector = db.query(func.count()).select_from(sector_q.subquery()).scalar()
n_all = db.query(func.count(distinct(TLS201_APPLN.docdb_family_id))).filter(*scope_filter(cfg)).scalar()
print(f"sector families (rule A): {n_sector:,} | all families in the window: {n_all:,}")


def code_lift(symbol_table, symbol_column, scheme):
    """Sector and overall family counts for each symbol found in the sector, with its lift.

    The blank-stripped symbol is computed in a subquery first and the counts group by
    that column: BigQuery does not accept GROUP BY on the replace() expression itself.
    """
    symbol = func.replace(symbol_column, " ", "").label("symbol")

    def family_symbol_pairs(*conditions):
        return (
            db.query(TLS201_APPLN.docdb_family_id.label("docdb_family_id"), symbol)
            .select_from(TLS201_APPLN)
            .join(symbol_table, TLS201_APPLN.appln_id == symbol_table.appln_id)
            .filter(*scope_filter(cfg), *conditions)
            .subquery()
        )

    sector_pairs = family_symbol_pairs(TLS201_APPLN.docdb_family_id.in_(sector_q.subquery().select()))
    n_sector_fam = func.count(distinct(sector_pairs.c.docdb_family_id))
    in_sector = (
        db.query(sector_pairs.c.symbol, n_sector_fam.label("sector_families"))
        .group_by(sector_pairs.c.symbol)
        .having(n_sector_fam >= MIN_CODE_FAMILIES)
        .subquery()
    )
    all_pairs = family_symbol_pairs(func.replace(symbol_column, " ", "").in_(db.query(in_sector.c.symbol).subquery().select()))
    overall = (
        db.query(all_pairs.c.symbol, func.count(distinct(all_pairs.c.docdb_family_id)).label("all_families"))
        .group_by(all_pairs.c.symbol)
        .subquery()
    )
    df = patstat.df(
        db.query(in_sector.c.symbol, in_sector.c.sector_families, overall.c.all_families)
        .join(overall, overall.c.symbol == in_sector.c.symbol)
    )
    columns = ["symbol", "sector_families", "all_families", "share_of_sector", "lift", "in_config"]
    if df.empty:                                     # patstat.df may return no columns at all
        return pd.DataFrame(columns=columns).astype({"in_config": bool})
    df["share_of_sector"] = (df["sector_families"] / n_sector).round(4)
    df["lift"] = ((df["sector_families"] / n_sector) / (df["all_families"] / n_all)).round(1)
    known = configured_codes(cfg, scheme)
    df["in_config"] = df["symbol"].map(lambda s: any(s.startswith(c) for c in known))
    return df.sort_values("lift", ascending=False)


cpc_lift = code_lift(TLS224_APPLN_CPC, TLS224_APPLN_CPC.cpc_class_symbol, "cpc")
ipc_lift = code_lift(TLS209_APPLN_IPC, TLS209_APPLN_IPC.ipc_class_symbol, "ipc")

for name, df in (("CPC", cpc_lift), ("IPC", ipc_lift)):
    print(f"\n=== {name}: codes NOT in the config, highest lift first (min {MIN_CODE_FAMILIES} sector families)")
    print(df.loc[~df["in_config"].astype(bool)].head(TOP).drop(columns="in_config").to_string(index=False))

**How to read it.** High lift **and** a reasonable number of sector families: likely a code we missed. Look up the title in the CPC scheme and decide: core (smart glasses by definition), component (needs an eyewear term), review, or ignore. Very high lift with few families is often a narrow subgroup of a code we already have.

## B · Words: keyness against random patents

**Sector documents:** title + English abstract of the rule-A families (from `03_texts`).
**Background:** a random sample of English abstracts of all patents in the window (applications with `appln_id` divisible by `BACKGROUND_MOD`).

In [ ]:
texts = read_table("texts")
texts = texts[texts["has_abstract"]].copy()
texts["doc"] = (texts["appln_title"].fillna("") + ". " + texts["appln_abstract"].fillna("")).str.lower()
sector_docs = texts.loc[texts["in_sector"], "doc"].tolist()
review_docs = texts.loc[texts["to_review"], "doc"].tolist()

background = patstat.df(
    db.query(TLS201_APPLN.docdb_family_id, TLS202_APPLN_TITLE.appln_title, TLS203_APPLN_ABSTR.appln_abstract)
    .join(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
    .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
    .filter(*scope_filter(cfg),
            func.lower(func.trim(TLS203_APPLN_ABSTR.appln_abstract_lg)) == "en",
            func.mod(TLS201_APPLN.appln_id, BACKGROUND_MOD) == 0)
    .limit(BACKGROUND_N)
)
background = background[~background["docdb_family_id"].isin(texts["docdb_family_id"])].drop_duplicates("docdb_family_id")
background_docs = (background["appln_title"].fillna("") + ". " + background["appln_abstract"].fillna("")).str.lower().tolist()
print(f"sector docs: {len(sector_docs):,} | background docs: {len(background_docs):,} | to-review docs: {len(review_docs):,}")

In [ ]:
vectorizer = CountVectorizer(
    ngram_range=(1, 3), stop_words="english", min_df=MIN_PHRASE_DF, binary=True,
    token_pattern=r"(?u)\b[a-z][a-z-]+\b",          # words of letters (and hyphens), no numbers
)
X = vectorizer.fit_transform(sector_docs + background_docs)
terms = vectorizer.get_feature_names_out()
n_s, n_b = len(sector_docs), len(background_docs)


def doc_share(matrix):
    """Share of documents (rows) containing each term."""
    return np.asarray(matrix.sum(axis=0)).ravel() / max(matrix.shape[0], 1)


share_sector = doc_share(X[:n_s])
share_background = doc_share(X[n_s:])
share_review = doc_share(vectorizer.transform(review_docs)) if review_docs else np.zeros(len(terms))

phrases = pd.DataFrame({
    "phrase": terms,
    "share_sector": share_sector.round(4),
    "share_background": share_background.round(5),
    "share_to_review": share_review.round(4),
})
# +1 smoothing so that phrases absent from the background get a finite keyness
phrases["keyness"] = (((share_sector * n_s + 1) / (n_s + 1)) / ((share_background * n_b + 1) / (n_b + 1))).round(1)

# A phrase that contains a specific keyword is already found by rule A
specific = [t.lower().strip() for t in sel["specific_keywords"]]
eyewear = [t.lower().strip() for t in sel["eyewear_terms"]]
phrases["already_covered"] = phrases["phrase"].map(lambda p: any(k in p for k in specific))
phrases["has_eyewear_term"] = phrases["phrase"].map(lambda p: any(k in p for k in eyewear))
phrases = phrases.sort_values("keyness", ascending=False)

MIN_SHARE_SECTOR = 0.005                        # in at least 0.5% of the sector documents
suggest_words = phrases[~phrases["already_covered"] & (phrases["share_sector"] >= MIN_SHARE_SECTOR)]
print(f"=== phrases NOT covered by the config, highest keyness first (in >= {MIN_SHARE_SECTOR:.1%} of sector docs)")
print(suggest_words.head(2 * TOP).drop(columns="already_covered").to_string(index=False))

**How to read it.**

- `share_sector` vs `share_background`: how much more common the phrase is in smart-glasses abstracts.
- `share_to_review`: how many of the doubtful families contain it. A good, specific phrase with a high value here would move many families from "to review" into the sector if added to `specific_keywords`.
- `has_eyewear_term`: the phrase already contains an eyewear word, so it is used by rule B with component codes; adding it to `specific_keywords` would make it enough on its own.
- Generic words that are only *frequent* (e.g. "display", "user") have low keyness; ambiguous ones (e.g. "glasses", "lens", "frame") are not specific enough alone: prefer longer phrases ("glasses body", "lens module").

## C · Audio & hearing words (Amplifon segment)

Sector families whose text mentions an audio / hearing term (`segments.audio_hearing.terms`), compared with the **rest of the sector**: which phrases characterise the audio segment.

In [ ]:
audio_terms = [t.lower() for t in cfg["segments"]["audio_hearing"]["terms"]]
is_audio = np.array([any(t in d for t in audio_terms) for d in sector_docs], dtype=bool)
print(f"audio / hearing sector docs: {is_audio.sum():,} of {n_s:,} ({is_audio.mean():.1%})")

X_sector = X[:n_s]
share_audio = doc_share(X_sector[is_audio])
share_other = doc_share(X_sector[~is_audio])
n_a, n_o = int(is_audio.sum()), int((~is_audio).sum())

audio_phrases = pd.DataFrame({"phrase": terms, "share_audio": share_audio.round(4), "share_other_sector": share_other.round(4)})
audio_phrases["keyness"] = (((share_audio * n_a + 1) / (n_a + 1)) / ((share_other * n_o + 1) / (n_o + 1))).round(1)
audio_phrases = audio_phrases[audio_phrases["share_audio"] >= 0.01].sort_values("keyness", ascending=False)
print("=== phrases typical of the audio / hearing segment")
print(audio_phrases.head(TOP).to_string(index=False))

## Save the suggestions

CSV files on TIP (`outputs/` is git-ignored). Share them with the group to review the candidates.

In [ ]:
cpc_lift.to_csv(OUTPUT_DIR / f"expansion_cpc_{ENV}.csv", index=False)
ipc_lift.to_csv(OUTPUT_DIR / f"expansion_ipc_{ENV}.csv", index=False)
phrases.to_csv(OUTPUT_DIR / f"expansion_phrases_{ENV}.csv", index=False)
audio_phrases.to_csv(OUTPUT_DIR / f"expansion_audio_phrases_{ENV}.csv", index=False)
print("saved expansion_cpc, expansion_ipc, expansion_phrases, expansion_audio_phrases in", OUTPUT_DIR)